# 1. Objetivo

Este notebook cobre a etapa de **Preparação para Machine Learning** do Tech Challenge, dando sequência ao `01_eda.ipynb`. A partir da camada Silver (já validada, limpa e enriquecida com as novas features comportamentais), aqui vamos:

- separar atributos (`X`) e variável alvo (`y`);
- construir os dois cenários de modelagem (`referencia` e `comportamental`);
- dividir cada cenário em treino/teste, garantindo que ambos usem exatamente a mesma divisão de linhas, para que a comparação entre os dois modelos na próxima etapa seja justa;
- construir e demonstrar o pré-processador (`ColumnTransformer`) de cada cenário.

Toda a lógica reutilizável está centralizada em `src/model_training.py`; este notebook apenas importa essas funções e documenta as decisões tomadas. **Nenhum modelo é treinado aqui** — o treinamento e a comparação entre os cenários ficam para o `03_modelo_baseline.ipynb`.

## 2. Importação das bibliotecas

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src.model_training import (
    RANDOM_STATE,
    TARGET,
    TEST_SIZE,
    carregar_dados_silver,
    criar_cenarios_modelagem,
    criar_preprocessador,
    dividir_treino_teste,
    obter_grupos_variaveis,
    separar_features_target,
)

## 3. Configurações iniciais

A semente (`RANDOM_STATE`), a proporção do conjunto de teste (`TEST_SIZE`) e o nome da variável alvo (`TARGET`) são definidos uma única vez em `src/model_training.py` e reutilizados em todos os notebooks de modelagem, garantindo que a mesma configuração seja aplicada de forma consistente do baseline até o modelo final.

In [2]:
print(f"RANDOM_STATE: {RANDOM_STATE}")
print(f"TEST_SIZE: {TEST_SIZE}")
print(f"TARGET: {TARGET}")

pd.set_option("display.max_columns", None)

RANDOM_STATE: 42
TEST_SIZE: 0.2
TARGET: ds_nivel_obesidade


## 4. Leitura da camada Silver

Carregamos a base já tratada na etapa de EDA (`data/silver/obesity_tratado.csv`) com `carregar_dados_silver()` e, em seguida, separamos os atributos preditores (`X`) da variável alvo (`y`) com `separar_features_target()`. Essa separação é feita uma única vez, antes da criação dos cenários, para que ambos os cenários derivem exatamente do mesmo `X`.

In [3]:
dados_silver = carregar_dados_silver()
dados_silver.shape

(2087, 21)

In [4]:
dados_silver.head()

,ds_genero,nr_idade,nr_altura,nr_peso,fl_historico_familiar_sobrepeso,fl_consumo_calorico_frequente,cd_consumo_de_vegetais,cd_numero_refeicoes_principais,ds_lanches_entre_refeicoes,fl_fumante,cd_consumo_agua,fl_monitora_calorias,cd_frequencia_atividade_fisica,cd_tempo_uso_eletronicos,ds_consumo_alcool,ds_meio_transporte,ds_nivel_obesidade,nr_imc,fl_transporte_ativo,ds_faixa_etaria,ds_consumo_alcool_agrupado
0,Female,21.0,1.62,64.0,1,0,2,3,Sometimes,0,2,0,0,1,no,Public_Transportation,Normal_Weight,24.386526,0,adulto_jovem,no
1,Female,21.0,1.52,56.0,1,0,3,3,Sometimes,1,3,1,3,0,Sometimes,Public_Transportation,Normal_Weight,24.238227,0,adulto_jovem,Sometimes
2,Male,23.0,1.80,77.0,1,0,2,3,Sometimes,0,2,0,2,1,Frequently,Public_Transportation,Normal_Weight,23.765432,0,adulto_jovem,consumo_frequente_ou_mais
3,Male,27.0,1.80,87.0,0,0,3,3,Sometimes,0,2,0,2,0,Frequently,Walking,Overweight_Level_I,26.851852,1,adulto_jovem,consumo_frequente_ou_mais
4,Male,22.0,1.78,89.8,0,0,2,1,Sometimes,0,2,0,0,0,Sometimes,Public_Transportation,Overweight_Level_II,28.342381,0,adulto_jovem,Sometimes


In [5]:
X, y = separar_features_target(dados_silver, target=TARGET)

print(f"X: {X.shape}")
print(f"y: {y.shape}")

X: (2087, 20)
y: (2087,)


## 5. Definição dos cenários de modelagem

Como discutido no `01_eda.ipynb`, o alvo deste dataset (`ds_nivel_obesidade`) é originalmente derivado de faixas de IMC (`peso / altura²`). Isso significa que `nr_altura`, `nr_peso` e o `nr_imc` calculado a partir deles quase determinam o alvo sozinhos — incluí-los sem ressalva tende a produzir um modelo com accuracy muito alta, mas que apenas reaprendeu a fórmula do IMC, sem valor clínico adicional sobre hábitos e estilo de vida.

Por isso, em vez de decidir unilateralmente incluir ou excluir essas colunas, `criar_cenarios_modelagem()` gera dois cenários explícitos, treinados e avaliados lado a lado:

- **`referencia`**: mantém todos os atributos, incluindo `nr_altura`, `nr_peso` e `nr_imc`. Funciona como um teto de performance e checagem de sanidade — se esse modelo não alcançar accuracy muito alta, algo está errado no pipeline, já que o alvo é quase determinístico em função dessas colunas.
- **`comportamental`**: remove `nr_altura`, `nr_peso` e `nr_imc`, restando apenas hábitos alimentares, atividade física e demais dados comportamentais/demográficos. É o modelo clinicamente relevante para o hospital, já que o objetivo de negócio é apoiar a equipe médica a partir de hábitos de vida, não recalcular o IMC que o paciente já tem.

Os dois cenários serão treinados e comparados no `03_modelo_baseline.ipynb`; a divergência de accuracy entre eles é, em si, uma evidência da circularidade do alvo.

In [6]:
cenarios = criar_cenarios_modelagem(X)

for nome_cenario, X_cenario in cenarios.items():
    print(f"Cenário '{nome_cenario}': {X_cenario.shape[1]} colunas -> {list(X_cenario.columns)}")

Cenário 'referencia': 20 colunas -> ['ds_genero', 'nr_idade', 'nr_altura', 'nr_peso', 'fl_historico_familiar_sobrepeso', 'fl_consumo_calorico_frequente', 'cd_consumo_de_vegetais', 'cd_numero_refeicoes_principais', 'ds_lanches_entre_refeicoes', 'fl_fumante', 'cd_consumo_agua', 'fl_monitora_calorias', 'cd_frequencia_atividade_fisica', 'cd_tempo_uso_eletronicos', 'ds_consumo_alcool', 'ds_meio_transporte', 'nr_imc', 'fl_transporte_ativo', 'ds_faixa_etaria', 'ds_consumo_alcool_agrupado']
Cenário 'comportamental': 17 colunas -> ['ds_genero', 'nr_idade', 'fl_historico_familiar_sobrepeso', 'fl_consumo_calorico_frequente', 'cd_consumo_de_vegetais', 'cd_numero_refeicoes_principais', 'ds_lanches_entre_refeicoes', 'fl_fumante', 'cd_consumo_agua', 'fl_monitora_calorias', 'cd_frequencia_atividade_fisica', 'cd_tempo_uso_eletronicos', 'ds_consumo_alcool', 'ds_meio_transporte', 'fl_transporte_ativo', 'ds_faixa_etaria', 'ds_consumo_alcool_agrupado']


## 6. Divisão treino/teste

Aplicamos `dividir_treino_teste()` separadamente para cada cenário — cada um tem um conjunto de colunas diferente, então a divisão precisa ser refeita para cada `X_cenario`. Como os dois cenários compartilham o mesmo `y`, o mesmo `RANDOM_STATE` e o mesmo `TEST_SIZE`, o split estratificado do `scikit-learn` seleciona os mesmos índices de linha em ambos os casos — mas isso é uma consequência da implementação, não algo que devemos apenas assumir. Por isso, validamos explicitamente que os índices de treino e de teste são idênticos entre os dois cenários antes de seguir adiante: essa igualdade é essencial para que a comparação entre os modelos de referência e comportamental, na próxima etapa, seja justa (mesmos pacientes em treino e em teste nos dois casos).

In [7]:
splits = {}

for nome_cenario, X_cenario in cenarios.items():
    X_train, X_test, y_train, y_test = dividir_treino_teste(
        X_cenario,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    splits[nome_cenario] = (X_train, X_test, y_train, y_test)
    print(f"Cenário '{nome_cenario}': X_train {X_train.shape}, X_test {X_test.shape}")

Cenário 'referencia': X_train (1669, 20), X_test (418, 20)
Cenário 'comportamental': X_train (1669, 17), X_test (418, 17)


In [8]:
X_train_referencia, X_test_referencia, y_train_referencia, y_test_referencia = splits["referencia"]
X_train_comportamental, X_test_comportamental, y_train_comportamental, y_test_comportamental = splits["comportamental"]

assert X_train_referencia.index.equals(X_train_comportamental.index), (
    "Os índices de treino divergem entre os cenários — a comparação entre os modelos não seria justa."
)
assert X_test_referencia.index.equals(X_test_comportamental.index), (
    "Os índices de teste divergem entre os cenários — a comparação entre os modelos não seria justa."
)
assert y_train_referencia.index.equals(y_train_comportamental.index)
assert y_test_referencia.index.equals(y_test_comportamental.index)

print("Confirmado: os cenários 'referencia' e 'comportamental' caíram exatamente na mesma divisão de linhas em treino/teste.")
print(f"Pacientes em treino: {len(X_train_referencia)} | Pacientes em teste: {len(X_test_referencia)}")

Confirmado: os cenários 'referencia' e 'comportamental' caíram exatamente na mesma divisão de linhas em treino/teste.
Pacientes em treino: 1669 | Pacientes em teste: 418


## 7. Construção e visualização do pré-processamento

Para cada cenário, usamos `obter_grupos_variaveis()` para classificar as colunas de `X_train` em três grupos (numéricas contínuas, ordinais e categóricas nominais) e `criar_preprocessador()` para montar o `ColumnTransformer` correspondente:

- numéricas contínuas recebem `StandardScaler`;
- ordinais (`cd_*`) passam sem transformação (`passthrough`), pois já representam escalas discretas arredondadas;
- categóricas nominais (`ds_*`) recebem One-Hot Encoding.

Ajustamos (`fit_transform`) cada pré-processador sobre o `X_train` do respectivo cenário — nunca sobre o conjunto completo, para evitar vazamento de informação do teste — e comparamos o formato antes/depois, incluindo quantas colunas o One-Hot Encoding gera a partir das categóricas.

### 7.1 Cenário `referencia`

In [9]:
grupos_referencia = obter_grupos_variaveis(X_train_referencia)

for grupo, colunas in grupos_referencia.items():
    print(f"{grupo} ({len(colunas)}): {colunas}")

numericas (9): ['nr_idade', 'nr_altura', 'nr_peso', 'fl_historico_familiar_sobrepeso', 'fl_consumo_calorico_frequente', 'fl_fumante', 'fl_monitora_calorias', 'nr_imc', 'fl_transporte_ativo']
ordinais (5): ['cd_consumo_de_vegetais', 'cd_numero_refeicoes_principais', 'cd_consumo_agua', 'cd_frequencia_atividade_fisica', 'cd_tempo_uso_eletronicos']
categoricas (6): ['ds_genero', 'ds_lanches_entre_refeicoes', 'ds_consumo_alcool', 'ds_meio_transporte', 'ds_faixa_etaria', 'ds_consumo_alcool_agrupado']


In [10]:
preprocessador_referencia = criar_preprocessador(X_train_referencia)

X_train_referencia_transformado = preprocessador_referencia.fit_transform(X_train_referencia)

colunas_ohe_referencia = preprocessador_referencia.named_transformers_["categoricas"].get_feature_names_out(
    grupos_referencia["categoricas"]
)

print(f"Antes do pré-processamento: {X_train_referencia.shape}")
print(f"Depois do pré-processamento: {X_train_referencia_transformado.shape}")
print(f"Colunas geradas pelo One-Hot Encoding ({len(colunas_ohe_referencia)}): {list(colunas_ohe_referencia)}")

Antes do pré-processamento: (1669, 20)
Depois do pré-processamento: (1669, 36)
Colunas geradas pelo One-Hot Encoding (22): ['ds_genero_Female', 'ds_genero_Male', 'ds_lanches_entre_refeicoes_Always', 'ds_lanches_entre_refeicoes_Frequently', 'ds_lanches_entre_refeicoes_Sometimes', 'ds_lanches_entre_refeicoes_no', 'ds_consumo_alcool_Always', 'ds_consumo_alcool_Frequently', 'ds_consumo_alcool_Sometimes', 'ds_consumo_alcool_no', 'ds_meio_transporte_Automobile', 'ds_meio_transporte_Bike', 'ds_meio_transporte_Motorbike', 'ds_meio_transporte_Public_Transportation', 'ds_meio_transporte_Walking', 'ds_faixa_etaria_adolescente', 'ds_faixa_etaria_adulto', 'ds_faixa_etaria_adulto_jovem', 'ds_faixa_etaria_meia_idade', 'ds_consumo_alcool_agrupado_Sometimes', 'ds_consumo_alcool_agrupado_consumo_frequente_ou_mais', 'ds_consumo_alcool_agrupado_no']


### 7.2 Cenário `comportamental`

In [11]:
grupos_comportamental = obter_grupos_variaveis(X_train_comportamental)

for grupo, colunas in grupos_comportamental.items():
    print(f"{grupo} ({len(colunas)}): {colunas}")

numericas (6): ['nr_idade', 'fl_historico_familiar_sobrepeso', 'fl_consumo_calorico_frequente', 'fl_fumante', 'fl_monitora_calorias', 'fl_transporte_ativo']
ordinais (5): ['cd_consumo_de_vegetais', 'cd_numero_refeicoes_principais', 'cd_consumo_agua', 'cd_frequencia_atividade_fisica', 'cd_tempo_uso_eletronicos']
categoricas (6): ['ds_genero', 'ds_lanches_entre_refeicoes', 'ds_consumo_alcool', 'ds_meio_transporte', 'ds_faixa_etaria', 'ds_consumo_alcool_agrupado']


In [12]:
preprocessador_comportamental = criar_preprocessador(X_train_comportamental)

X_train_comportamental_transformado = preprocessador_comportamental.fit_transform(X_train_comportamental)

colunas_ohe_comportamental = preprocessador_comportamental.named_transformers_["categoricas"].get_feature_names_out(
    grupos_comportamental["categoricas"]
)

print(f"Antes do pré-processamento: {X_train_comportamental.shape}")
print(f"Depois do pré-processamento: {X_train_comportamental_transformado.shape}")
print(f"Colunas geradas pelo One-Hot Encoding ({len(colunas_ohe_comportamental)}): {list(colunas_ohe_comportamental)}")

Antes do pré-processamento: (1669, 17)
Depois do pré-processamento: (1669, 33)
Colunas geradas pelo One-Hot Encoding (22): ['ds_genero_Female', 'ds_genero_Male', 'ds_lanches_entre_refeicoes_Always', 'ds_lanches_entre_refeicoes_Frequently', 'ds_lanches_entre_refeicoes_Sometimes', 'ds_lanches_entre_refeicoes_no', 'ds_consumo_alcool_Always', 'ds_consumo_alcool_Frequently', 'ds_consumo_alcool_Sometimes', 'ds_consumo_alcool_no', 'ds_meio_transporte_Automobile', 'ds_meio_transporte_Bike', 'ds_meio_transporte_Motorbike', 'ds_meio_transporte_Public_Transportation', 'ds_meio_transporte_Walking', 'ds_faixa_etaria_adolescente', 'ds_faixa_etaria_adulto', 'ds_faixa_etaria_adulto_jovem', 'ds_faixa_etaria_meia_idade', 'ds_consumo_alcool_agrupado_Sometimes', 'ds_consumo_alcool_agrupado_consumo_frequente_ou_mais', 'ds_consumo_alcool_agrupado_no']


### 7.3 Comparação entre os cenários

In [13]:
resumo_preprocessamento = pd.DataFrame(
    {
        "Cenário": ["referencia", "comportamental"],
        "Colunas em X_train (antes)": [X_train_referencia.shape[1], X_train_comportamental.shape[1]],
        "Colunas após pré-processamento": [
            X_train_referencia_transformado.shape[1],
            X_train_comportamental_transformado.shape[1],
        ],
        "Colunas geradas pelo One-Hot": [len(colunas_ohe_referencia), len(colunas_ohe_comportamental)],
    }
)

resumo_preprocessamento

,Cenário,Colunas em X_train (antes),Colunas após pré-processamento,Colunas geradas pelo One-Hot
0,referencia,20,36,22
1,comportamental,17,33,22


## 8. Conclusões

- A base Silver foi carregada e separada em `X`/`y`, servindo de ponto de partida único para os dois cenários de modelagem.
- Dois cenários foram definidos de forma explícita e documentada: **`referencia`** (com `nr_altura`, `nr_peso` e `nr_imc`, como teto de performance) e **`comportamental`** (sem essas três colunas, como modelo clinicamente relevante para o hospital).
- Os dois cenários foram divididos em treino/teste de forma independente, mas a igualdade de índices foi validada explicitamente (via `assert`) — treino e teste contêm exatamente os mesmos pacientes nos dois cenários, o que é pré-requisito para comparar os modelos de forma justa no próximo notebook.
- O pré-processador de cada cenário foi construído e ajustado sobre o respectivo `X_train`: variáveis numéricas contínuas foram padronizadas, ordinais mantidas como estão e categóricas nominais expandidas via One-Hot Encoding. O cenário `comportamental` gera uma matriz com menos colunas de entrada (por não ter `nr_altura`/`nr_peso`/`nr_imc`), mas o mesmo número de colunas geradas pelo One-Hot Encoding, já que as variáveis categóricas nominais são as mesmas nos dois cenários.

Próximo passo (`03_modelo_baseline.ipynb`): treinar múltiplos algoritmos com validação cruzada estratificada para os dois cenários, usando os pré-processadores construídos aqui dentro de uma `Pipeline` (`criar_pipeline`), e comparar as métricas (accuracy, F1-macro e matriz de confusão) entre `referencia` e `comportamental`.